In [32]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

In [33]:
demographic = pd.read_csv('cleaned dataset/api_data_aadhar_demographic.csv')


display(demographic.head())

,Unnamed: 0,date,state,district,pincode,age between 5 and 17,age 17 and above
0,0,2025-03-01,uttar pradesh,gorakhpur,273213,49,529
1,1,2025-03-01,andhra pradesh,chittoor,517132,22,375
2,2,2025-03-01,gujarat,rajkot,360006,65,765
3,3,2025-03-01,andhra pradesh,srikakulam,532484,24,314
4,4,2025-03-01,rajasthan,udaipur,313801,45,785


In [34]:
# work_hub = [
#     'bngaluru uban', 'mumbai city', 'mumbai suburban', 'new delhi ',
#     'south delhi', 'north delhi', 'hyderabad', 'chennai', 'kolkata', 'pune',
#     'ahmedabad', 'surat', 'gautam buddha nagar', 'gurugram', 'visakhapatnam',
#     'indore', 'coimbatore', 'vadodara', 'ernakulam', 'khordha', 'kamrup', 
#     'chandigarh', 'dakshina kannada', 'mysuru', 'ludhiana', 'nashik', 
#     'aurangabad', 'faridabad', 'ghaziabad', 'nagpur', 'jaipur', 'lucknow', 
#     'bhopal', 'patna', 'raipur', 'dehradun', 'thiruvananthapuram'
# ]

work_hubs = [
    'bengaluru urban', 'mumbai city', 'mumbai suburban', 'new delhi', 'south delhi', 
    'pune', 'surat', 'ahmedabad', 'gurugram', 'hyderabad', 'chennai', 'kolkata'
]

df = demographic.copy()

df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

In [35]:
def work_hub (data):
    if data in work_hubs:
        return 'work hub'
    else:
        return 'source/rural'

df['District_Type'] = df['district'].apply(work_hub)

display(df.head())

,Unnamed: 0,date,state,district,pincode,age between 5 and 17,age 17 and above,District_Type
0,0,2025-03-01,uttar pradesh,gorakhpur,273213,49,529,source/rural
1,1,2025-03-01,andhra pradesh,chittoor,517132,22,375,source/rural
2,2,2025-03-01,gujarat,rajkot,360006,65,765,source/rural
3,3,2025-03-01,andhra pradesh,srikakulam,532484,24,314,source/rural
4,4,2025-03-01,rajasthan,udaipur,313801,45,785,source/rural


In [36]:
table = df.groupby(['date', 'District_Type', 'district'])['age 17 and above'].sum().reset_index()

table.rename(columns={'age 17 and above': 'Migration_Volume'}, inplace=True)

display(table.head())

,date,District_Type,district,Migration_Volume
0,2025-03-01,source/rural,adilabad,12090
1,2025-03-01,source/rural,agar malwa,1966
2,2025-03-01,source/rural,agra,31323
3,2025-03-01,source/rural,ahilyanagar,27685
4,2025-03-01,source/rural,aizawl,5276


In [37]:
# We calculate Mean and Std Dev for EACH district separately
district_stats = table.groupby('district')['Migration_Volume'].agg(['mean', 'std']).reset_index()

print(district_stats.head())
table = table.merge(district_stats, on='district')

print(table.head())

      district         mean          std
0     adilabad   705.234043  1254.052934
1   agar malwa    95.903226   231.436913
2         agra  1693.290323  3261.903264
3  ahilyanagar  2494.131868  3536.265463
4    ahmedabad  2564.126316  6164.522636
        date District_Type     district  Migration_Volume         mean  \
0 2025-03-01  source/rural     adilabad             12090   705.234043   
1 2025-03-01  source/rural   agar malwa              1966    95.903226   
2 2025-03-01  source/rural         agra             31323  1693.290323   
3 2025-03-01  source/rural  ahilyanagar             27685  2494.131868   
4 2025-03-01  source/rural       aizawl              5276   124.824176   

           std  
0  1254.052934  
1   231.436913  
2  3261.903264  
3  3536.265463  
4   547.887409  


In [38]:
# Calculate Z-Score: How many Standard Deviations is today's volume away from normal?
# Formula: (Today - Mean) / StdDev

table['Migration_Z_Score'] = (table['Migration_Volume'] - table['mean']) / (table['std'] + 1) # +1 avoids division by zero

In [39]:
# --- 5. Add Season Context ---
def get_harvest_season(date_obj):
    m = date_obj.month
    if m in [4, 5]: return "Rabi Harvest (Labor leaves City)"
    elif m in [6, 7]: return "Post-Rabi Return (Labor returns to City)"
    elif m in [10, 11]: return "Kharif Harvest (Labor leaves City)"
    elif m in [12, 1]: return "Post-Kharif Return (Labor returns to City)"
    else: return "Lean Season"

table['Season'] = table['date'].apply(get_harvest_season)

In [41]:
# We only care if the Z-Score is > 2 (Significant Spike)

significant_migration = table[table['Migration_Z_Score'] > 2].copy()
significant_migration.sort_values(by='Migration_Z_Score', ascending=False, inplace=True)


district_means = significant_migration.groupby('district')['Migration_Volume'].transform('mean')
significant_migration['Spike_Intensity'] = significant_migration['Migration_Volume'] - district_means

display(significant_migration.head())

,date,District_Type,district,Migration_Volume,mean,std,Migration_Z_Score,Season,Spike_Intensity
666,2025-03-01,source/rural,udaipur,58055,1247.915789,5910.721097,9.609229,Lean Season,0.0
59,2025-03-01,source/rural,baran,39435,761.947368,4035.591924,9.580620,Lean Season,0.0
104,2025-03-01,source/rural,bundi,21657,472.180851,2221.030909,9.533989,Lean Season,0.0
1389,2025-06-01,source/rural,mahasamund,67874,1329.410526,6988.707314,9.520369,Post-Rabi Return (Labor returns to City),0.0
283,2025-03-01,source/rural,jhalawar,45544,804.956989,4701.001001,9.514894,Lean Season,0.0


In [ ]:
# Export
significant_migration.to_csv('csv for power bi/migration_analysis_corrected.csv', index=False)

Metric Interpretation:

Positive Number: Migration is higher than average for that district (An influx event).

Negative Number: Migration is lower than average.

Visualization:

Plot Spike_Intensity on the Y-Axis.

Plot Date on the X-Axis.

Use a Clustered Column Chart.

Conditional Formatting: Set columns > 0 to Red (High Influx) and < 0 to Gray.

Look for the Red Bars that align with your "Harvest Cycle" legend.